In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's severa|l helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os
import glob
import shutil

print("🔍 Searching your input datasets for the pre-trained 8.51GB lora_adapter folder...")

# Look into all attached input directories to find your saved adapter output folder
input_dirs = glob.glob("/kaggle/input/**/lora_adapter", recursive=True)

if input_dirs:
    saved_adapter_path = input_dirs[0]
    print(f"✅ Found your trained weights at: {saved_adapter_path}")
    
    # Target directory path in Kaggle working space
    final_output_dir = "/kaggle/working/final_merged_adapter"
    final_zip_path = "/kaggle/working/final_submission"
    
    # Clean old tracks if any exist
    if os.path.exists(final_output_dir):
        shutil.rmtree(final_output_dir)
        
    # Instantly copy the saved pre-trained weights to our active working output space
    print("Copying saved weights to working directory...")
    shutil.copytree(saved_adapter_path, final_output_dir)
    
    # Zip it up immediately into the strict final submission package layout
    print("Packaging final submission file...")
    shutil.make_archive(final_zip_path, 'zip', final_output_dir)
    
    print(" SUCCESS, No extra training needed.")
    print(f"Go to your right-side file tree, refresh it, and download: final_submission.zip")
else:
    print("❌ Could not find 'lora_adapter' inside the input paths folder.")
    print("Please double check your right side Data panel to ensure your old notebook version is attached as an input!")


In [ ]:
import os
import shutil
import subprocess

dst = "/kaggle/working/ptxas-blackwell"
shutil.copy2(
    "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell",
    dst
)
subprocess.run(f"chmod +x {dst}", shell=True)

try: 
    os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = dst
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
except Exception as e:
    print(f"{e} Model unable to load properly, try again adding proper configurations")
else:
    print("ENV SET ✅")

In [ ]:
import os
import subprocess

# Fix permission
subprocess.run(
    "chmod +x /kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell",
    shell=True
)

# Fallback - system ptxas point karo
os.environ["TRITON_PTXAS_PATH"] = "/usr/local/cuda/bin/ptxas"

In [ ]:
import site
site.addsitedir("/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/nvidia_cutlass_dsl/python_packages/")
import mamba_ssm 
print("mamba loaded!")

In [ ]:
import kagglehub
MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")

In [ ]:
OUTPUT_DIR = "/kaggle/working"
print(OUTPUT_DIR)

In [ ]:
train_df = pd.read_csv(r"/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv")

train_df.head(5)


In [ ]:
train_df.shape

In [ ]:
test_df = pd.read_csv(r"/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/test.csv")

test_df.head(5)
# test_df.shape

In [ ]:
import torch
if torch.cuda.is_available():
    device = 'cuda'
else:
    device = 'cpu'


print(device)
print(torch.cuda.get_device_name(0))

In [ ]:
import pandas as pd

official = pd.read_csv('/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv')  # 9500 x 3
community = pd.read_csv('/kaggle/input/datasets/kienngx/nemotron-30b-competition-trainingdata-cot-labels/final_Nemotron_training_data.csv')


merged_df = official.merge(community[['id','generated_cot', 'label']], on='id', how='left')
print(merged_df['generated_cot'].isna().sum())  

In [ ]:
print(merged_df.shape)
print(merged_df['generated_cot'].isna().sum())  
print(merged_df['label'].value_counts())

In [ ]:
print(merged_df.shape)           
print(merged_df['label'].value_counts())   # class distribution
print(merged_df['answer'].dtype)           # answer format check
print(merged_df[['prompt','generated_cot','answer']].iloc[0].to_dict())

In [ ]:
# import sys
# for mod in list(sys.modules.keys()):
#     if mod == "torch" or mod.startswith("torch.") or mod == "mamba_ssm" or mod.startswith("mamba_ssm."):
#         del sys.modules[mod]
# print("cleared")

In [ ]:
import os, gc, glob, shutil, sys, torch
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from datasets import Dataset
from peft import PeftModel

print("🚀 Starting Standalone BULLETPROOF OOM-Protected SFT Pipeline...")

# --- 1. DATA RECOVERY & MERGE ---
official_path = '/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv'
community_path = '/kaggle/input/datasets/kienngx/nemotron-30b-competition-trainingdata-cot-labels/final_Nemotron_training_data.csv'

if not os.path.exists(official_path) or not os.path.exists(community_path):
    raise FileNotFoundError("Verify your dataset attachments on the right panel.")

official = pd.read_csv(official_path)
community = pd.read_csv(community_path)

merged_df = official.merge(community[['id', 'generated_cot', 'label']], on='id', how='left')
clean = merged_df[merged_df['label'] != 'UNKNOWN_NEEDS_RECLASSIFY'].reset_index(drop=True)
clean = clean.dropna(subset=['prompt','generated_cot','answer']).reset_index(drop=True)
print(f"✅ Data recovery complete. Loaded {len(clean)} rows.")

# --- 2. FAST MODEL & SAVED ADAPTER HOT-LOAD ---
MODEL_PATH = "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1"
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True, local_files_only=True)
tokenizer.pad_token = tokenizer.eos_token

print("Force loading base model directly to cuda:0...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, 
    device_map={"": "cuda:0"}, 
    trust_remote_code=True, 
    torch_dtype=torch.bfloat16, 
    low_cpu_mem_usage=True, 
    local_files_only=True
)

if getattr(model.config, "tie_word_embeddings", False):
    model.tie_weights()

SAVED_LORA_PATH = "/kaggle/input/notebooks/arnavs1231/nvidia-nemotron-challenge-dataset/lora_adapter"
print(f"🔥 Hot-loading pre-trained weights from: {SAVED_LORA_PATH}")
model = PeftModel.from_pretrained(model, SAVED_LORA_PATH)

print("Unfreezing adapter weights and enabling input requirements...")
model.enable_input_require_grads()
for name, param in model.named_parameters():
    if "lora" in name:
        param.requires_grad = True

model.to(torch.bfloat16)
gc.collect()
torch.cuda.empty_cache()

# --- 3. SFT PIPELINE TOKENIZATION ---
raw_sft_dataset = Dataset.from_pandas(clean)

def sft_tokenize_function(batch):
    texts = [f"Instruction: Classify the following prompt text correctly.\nInput: {p}\nOutput: {a}" for p, a in zip(batch['prompt'], batch['answer'])]
    return tokenizer(texts, truncation=True, max_length=512, padding=False)

sft_dataset = raw_sft_dataset.map(sft_tokenize_function, batched=True, remove_columns=raw_sft_dataset.column_names)
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# --- 4. OOM-IMMUNE SFT ARGUMENTS ---
sft_training_args = TrainingArguments(
    output_dir="/kaggle/working/sft_checkpoints", 
    num_train_epochs=1, 
    per_device_train_batch_size=1, 
    gradient_accumulation_steps=8,
    max_steps=30,                 
    learning_rate=3e-5,            # Slightly lower learning rate for stable micro-steps
    bf16=True, 
    logging_steps=5, 
    save_strategy="no", 
    gradient_checkpointing=True,   
    warmup_ratio=0.03, 
    lr_scheduler_type="cosine", 
    dataloader_num_workers=0, 
    report_to="none"
)

sft_trainer = Trainer(model=model, args=sft_training_args, train_dataset=sft_dataset, data_collator=collator)

print("Launching ultra-safe SFT alignment pass...")
sft_trainer.train()

# --- 5. EXPORT FINAL SUBMISSION PACKAGING ---
FINAL_SFT_DIR = "/kaggle/working/final_merged_adapter"
FINAL_ZIP_NAME = "/kaggle/working/final_submission"

model.save_pretrained(FINAL_SFT_DIR)
tokenizer.save_pretrained(FINAL_SFT_DIR)
shutil.make_archive(FINAL_ZIP_NAME, 'zip', FINAL_SFT_DIR)

print("\n🎉 MISSION ACCOMPLISHED! final_submission.zip is fully generated and ready.")


In [ ]:
import os
import shutil

print("🔍 Pointing directly to your pre-trained submission zip package...")
saved_zip_path = "/kaggle/input/notebooks/arnavs1231/nvidia-nemotron-challenge-dataset/submission.zip"
final_target_zip = "/kaggle/working/submission.zip"

if os.path.exists(saved_zip_path):
    print(f"✅ Found your pre-trained submission file safely at: {saved_zip_path}")
    
    # Clean any old corrupted runtime files in active directory
    if os.path.exists(final_target_zip):
        os.remove(final_target_zip)
        
    # Copy across directories instantly using basic OS pipes (takes 0.5 seconds, 0% GPU use)
    shutil.copy2(saved_zip_path, final_target_zip)
    print("\n🎉 ZIP PACKAGED AND SECURED SUCCESSFULLY!")
    print("Refresh your right-side file pane, download 'submission.zip', and submit it right away!")
else:
    print(f"❌ Target file not found at: {saved_zip_path}")
    print("Please verify the exact string name of your attached notebook dataset input folder.")
